In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Parameters, State, and Memory

The previous section built models as trees of modules and found every
parameter by walking the tree. So far the framework has handled those
tensors for us, and we have not asked what else a model holds or how much
memory training it takes. Both questions matter once a model is large.
Consider a model with 8 billion parameters, the size of the Llama 3 model
that that section mentioned. Its weights take 32 GB in
the 32-bit floating-point format we have used so far. Training it takes four
times as much memory, 128 GB, before it has seen a single example: every
weight needs a gradient of the same size, and the Adam optimizer, which we
meet in that section, keeps two more tensors of that size, running
averages of the gradients and of their squares. That is more than an 80 GB
GPU holds.

To estimate such numbers we need to know every tensor a model holds and what
each one is for. Other routine operations need the same knowledge: a
checkpoint must save every tensor the computation depends on, a move to a
GPU must take all of them along, and fine-tuning must decide which of them
the optimizer may change.

Frameworks make this possible by *registration*: each layer announces the
tensors it owns to the module system, which files them by name in the tree
of modules of that section. The optimizer, the
checkpoint, and the device move then all walk the same tree. This section
follows the tree to its leaves, separates the tensors the optimizer trains
from those it must leave alone, counts the bytes that training stores per
parameter, and lets two layers share one tensor or excludes part of a model
from training.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The State Tree

### Access by Path

Our specimen is a residual MLP: an input layer maps 20 features to 64, two
residual blocks of width 64 follow, and an output layer produces 10 outputs.
The blocks are the residual block of that section,
defined again here so that this section stands on its own: each adds the
output of a small two-layer network, its *body*, to its input.

In [2]:
class ResidualBlock(nnx.Module):
    def __init__(self, num_hiddens, *, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

rngs = nnx.Rngs(d2l.get_key())
net = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                     ResidualBlock(64, rngs=rngs),
                     ResidualBlock(64, rngs=rngs),
                     nnx.Linear(64, 10, rngs=rngs))
X = jax.random.normal(rngs(), (2, 20))
net(X).shape

(2, 10)

A model built from NNX modules is an object graph, and its parameters are
`nnx.Param` variables owned by the layers. To reach one, follow ordinary
attributes and container indices. `net.layers[3]` is the output layer, and
its `bias` variable is marked trainable by its type.

In [3]:
bias = net.layers[3].bias
type(bias), bias.shape

(flax.nnx.variablelib.Param, (10,))

The same path syntax reaches arbitrarily deep. The first linear layer inside
the first residual block sits three levels down:

In [4]:
net.layers[1].body.layers[0].kernel.shape

(64, 64)

Gradients are not stored on parameters. `nnx.grad` returns a second state
tree with the same paths, one gradient leaf per parameter:

In [5]:
params = nnx.state(net, nnx.Param)
grads = nnx.grad(lambda model: model(X).sum())(net)
jax.tree_util.tree_structure(grads) == jax.tree_util.tree_structure(params)

True

Reaching parameters one path at a time suits debugging a single layer. The
optimizer and the checkpoint instead need every leaf, and
`nnx.state(net, nnx.Param)` selects them. Its flat view yields each array
together with the path that reaches it.

The model has 12 parameter tensors. We print their number and the four
that belong to the first residual block:

In [6]:
flat = list(params.flat_state())
len(flat), [(path, leaf.shape) for path, leaf in flat[2:6]]

(12,
 [(('layers', 1, 'body', 'layers', 0, 'bias'), (64,)),
  (('layers', 1, 'body', 'layers', 0, 'kernel'), (64, 64)),
  (('layers', 1, 'body', 'layers', 2, 'bias'), (64,)),
  (('layers', 1, 'body', 'layers', 2, 'kernel'), (64, 64))])

Read one of the paths closely.
`('layers', 1, 'body', 'layers', 0, 'kernel')` means: child 1 of `net` (the
first residual block), its `body`, child 0 of that sequence, and finally its
kernel. Paths survive any amount of nesting. Without a filter, `nnx.state`
returns all of the model's state, the tree that a checkpoint saves
(that section). For this model it is the parameter tree:

In [7]:
([path for path, _ in nnx.state(net).flat_state()]
 == [path for path, _ in params.flat_state()])

True

Every consumer walks this one tree under these names. The equality above
holds for this model because all of its state is trainable. That is not
always so.

### Parameters and Buffers

Suppose a model standardizes its inputs with a mean and a standard deviation
computed once from training data. These numbers must be saved with the model,
since predictions are wrong without them, and they must move to the GPU with
it. Yet the optimizer must never change them: they are statistics of the
data, not quantities to fit. Batch normalization [@Ioffe.Szegedy.2015],
a layer that we study in that section, keeps tensors of the same
kind: a running mean and variance of its inputs, updated during training and
used at prediction time. Other tensors live only for one call, such as the
intermediate values that backpropagation stores during a training step
(that section). They belong to the computation, not to the model,
and are neither saved nor registered.

NNX marks the kind of each piece of state by a `Variable` subclass:
`nnx.Param` for trainable state, `nnx.BatchStat` for running statistics, and
a subclass of our own for any other persistent tensor. Filters then select
the kinds that a consumer sees. The rule: make a tensor an `nnx.Param` if
the optimizer should update it, another `Variable` if checkpoints need it,
and a local variable if it lives for one call. Plain Python attributes hold
architecture constants such as layer widths. Here is a module that
standardizes its inputs with statistics computed from a reference sample:

In [8]:
class Buffer(nnx.Variable):
    pass

class Whitener(nnx.Module):
    def __init__(self, mean, std, *, rngs):
        self.mean = Buffer(mean)
        self.std = Buffer(std)
        self.out = nnx.Linear(4, 2, rngs=rngs)

    def __call__(self, X):
        return self.out((X - self.mean) / self.std)

sample = jax.random.normal(rngs(), (100, 4)) * jnp.arange(1., 5.)
whiten = Whitener(sample.mean(0), sample.std(0), rngs=rngs)
[(path, value.shape) for path, value in nnx.state(whiten).flat_state()]

[(('mean',), (4,)),
 (('out', 'bias'), (2,)),
 (('out', 'kernel'), (4, 2)),
 (('std',), (4,))]

Both kinds belong to the object graph and are checkpointed together. An
optimizer built over `nnx.Param` never sees the statistics:

In [9]:
{'params': list(nnx.state(whiten, nnx.Param)),
 'buffers': list(nnx.state(whiten, Buffer))}

{'params': ['out'], 'buffers': ['mean', 'std']}

Wrapping an array in a `Variable` makes it state, and operations on the
state tree reach it. We can see this on the CPU by splitting the model into
its structure and its state, converting every state leaf to another *dtype*,
and merging the result into a low-precision clone. The buffers convert with
the weights, and the original model is unchanged:

In [10]:
graph, state = nnx.split(whiten)
half = jax.tree.map(lambda x: x.astype(jnp.float16), state)
low = nnx.merge(graph, half)
low.mean.dtype, whiten.mean.dtype

(dtype('float16'), dtype('float32'))

Device placement follows the same rule: `jax.device_put` moves a whole state
tree, parameters and buffers alike, and updating the model with the moved
tree moves all registered state. The same lines run on any machine; on one
with an accelerator, `jax.devices()[0]` is a GPU:

In [11]:
moved = jax.device_put(nnx.state(whiten), jax.devices()[0])
nnx.update(whiten, moved)
whiten.mean.device

CudaDevice(id=0)

## Training Memory

### Bytes per Parameter

How much memory does training our specimen take? For the parameters and
what the optimizer stores about them, the answer follows from three counts
per parameter: the weight itself, its gradient, and the optimizer's state
for it. Minibatch gradient descent, which we have used so far, keeps no
state. Adam keeps two numbers per parameter, running averages of the
gradient and of its square, called its first and second *moments*. With
every tensor in 32-bit floating point (fp32), 4 bytes per number, training
with Adam stores

| Training state       | Precision | Bytes per parameter |
|----------------------|-----------|---------------------|
| Weights              | fp32      | 4                   |
| Gradients            | fp32      | 4                   |
| Adam first moment    | fp32      | 4                   |
| Adam second moment   | fp32      | 4                   |
| Total                |           | 16                  |

These 16 bytes are a floor. They omit the *activations*, the intermediate
values that backpropagation keeps from the forward pass, whose size grows
with the batch; they also omit temporary workspaces and the memory the
allocator holds in reserve. that section measures all of these on a
GPU. For our specimen, the floor is:

In [12]:
leaves = jax.tree_util.tree_leaves(nnx.state(net, nnx.Param))
n = sum(x.size for x in leaves)
weights = sum(x.size * x.dtype.itemsize for x in leaves)   # fp32: 4 bytes
grads, adam_state = weights, 2 * weights
print(f'{n} parameters: {(weights + grads + adam_state) / 2**20:.2f} MiB '
      'for weights + gradients + Adam state')

18634 parameters: 0.28 MiB for weights + gradients + Adam state


The count checks by hand: the input layer has $20 \times 64 + 64 = 1344$
parameters, each residual block $2 \times (64 \times 64 + 64) = 8320$, and
the output layer $64 \times 10 + 10 = 650$, for a total of 18,634. Adam's
state is real memory, allocated tensor by tensor. After one training step,
its two moments hold two more copies of the model:

In [13]:
optimizer = nnx.Optimizer(net, optax.adam(1e-3), wrt=nnx.Param)
grads = nnx.grad(lambda model: model(X).sum())(net)
optimizer.update(net, grads)
moments = sum(x.size for x in jax.tree.leaves(optimizer.opt_state)
              if hasattr(x, 'ndim') and x.ndim > 0)
moments == 2 * n

True

For our network the floor is 0.28 MiB, which is why memory has not
mattered so far. For the 8-billion-parameter model of the opening, the same
arithmetic gives 32 GB for the weights and 128 GB for weights, gradients,
and Adam's state, whatever the batch size. *Mixed-precision* training, the
subject of that section, computes with 16-bit numbers but does not
shrink this floor. A common recipe stores the weights and the gradients in
16 bits for the forward and backward passes, and keeps an fp32 master copy
of the weights and fp32 moments, so that small updates are not rounded away
[@Micikevicius.Narang.Alben.ea.2018]: $2 + 2 + 4 + 4 + 4 = 16$ bytes
per parameter, the same floor. Automatic casting in the frameworks keeps the
weights and gradients in fp32 and makes 16-bit copies operation by
operation, which also stores 16 bytes per parameter. A multiplier therefore
means little until we know the dtype of each tensor.

### Counting Without Allocating

Every count so far read the shapes of tensors that already existed. For the
8B model this order is backwards: to learn whether its 32 GB of weights fit
in memory, we would first have to allocate them. The count needs only the
shape and the dtype of each parameter, and both follow from the arguments of
the constructors. Take one layer of the 8B model, its output layer, which
maps a hidden state of width 4,096 to scores for the 128,256 tokens of its
vocabulary. Its weight has $4096 \times 128{,}256 = 525{,}336{,}576$
entries, 2.1 GB in fp32. We create the layer without its data and count.

`nnx.eval_shape` runs a constructor abstractly: each array operation in it
computes only the shape and the dtype of its result. The module it returns
holds a `jax.ShapeDtypeStruct` in place of each array:

In [ ]:
key = d2l.get_key()   # drawn outside the abstract trace
head = nnx.eval_shape(lambda: nnx.Linear(
    4096, 128256, use_bias=False, rngs=nnx.Rngs(key)))
leaves = jax.tree.leaves(nnx.state(head, nnx.Param))
n_head = sum(x.size for x in leaves)
print(leaves, f'{n_head:,} params, {4 * n_head / 1e9:.1f} GB in fp32')

No array was created and no random number drawn. Calling the constructor
itself, outside `nnx.eval_shape`, allocates the arrays and draws their
initial values.

Applied to every layer of the 8B model, the same count gives its 8 billion
parameters, and with them the 32 GB of weights and the 128 GB of training
state, without allocating a byte.

## Shared, Frozen, and Averaged Parameters

### Tied Parameters

One tensor can serve several roles in a model. The standard example is a
language model, which reads a sequence of *tokens* (words or pieces of words
from a vocabulary $V$) and predicts the next one (that section).
Its first layer is an *embedding*, a $|V| \times d$ table whose row $v$ is
the $d$-dimensional vector that represents token $v$. Its output layer maps a
$d$-dimensional hidden state to $|V|$ scores, one per token, so its weight
matrix has the same shape and an analogous meaning: one vector per token.
*Tying* the two roles to a single tensor saves $|V| \times d$ parameters, and
the studies that introduced it found that it also improves the predictions
[@Press.Wolf.2017; @Inan.Khosravi.Socher.2017]. The savings are large: in
GPT-2 small [@Radford.Wu.Child.ea.2019], the shared
$50257 \times 768$ table holds about 38.6 million of the model's 124 million
parameters, roughly 31%.

![One weight matrix W, two call sites: the embedding looks up a row by token id, the output head multiplies by its transpose to produce logits. Tying means both point at the same tensor, so gradients from both uses sum into one place.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-weight-tying.svg)

the figure shows the two call sites. A toy language model,
`ToyLM`, with a vocabulary of 100 tokens and $d = 16$, shows the mechanics.

`nnx.Embed` provides `attend`, which multiplies a hidden state by the
transpose of the embedding table, the output projection of a tied model.
Calling it reuses the module's one table, so the tying is visible in the
state itself: no head matrix exists.

In [15]:
class ToyLM(nnx.Module):
    def __init__(self, vocab_size, d, tied=True, *, rngs):
        self.tied = tied
        self.emb = nnx.Embed(vocab_size, d, rngs=rngs)
        self.body = nnx.Linear(d, d, rngs=rngs)
        self.head = None if tied else nnx.Linear(
            d, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, tokens):
        h = nnx.relu(self.body(self.emb(tokens)))
        if self.tied:
            return self.emb.attend(h)   # the embedding table, used as the head
        return self.head(h)

lm = ToyLM(vocab_size=100, d=16, rngs=rngs)
tokens = jax.random.randint(rngs(), (2, 8), 0, 100)
[(path, value.shape)
 for path, value in nnx.state(lm, nnx.Param).flat_state()]

[(('body', 'bias'), (16,)),
 (('body', 'kernel'), (16, 16)),
 (('emb', 'embedding'), (100, 16))]

Because the state contains the table once, there is no aliasing to keep
track of. The parameter count shows the $|V| \times d$ saving, and an
optimizer updates the table once per step:

In [16]:
untied = ToyLM(vocab_size=100, d=16, tied=False, rngs=rngs)
(sum(x.size for x in jax.tree.leaves(nnx.state(lm, nnx.Param))),
 sum(x.size for x in jax.tree.leaves(nnx.state(untied, nnx.Param))))

(1872, 3472)

The tied model has $1600 + 272 = 1872$ parameters: the table and the body
layer. The untied one adds a second $100 \times 16$ table, 3472 in all.

A checkpoint of the tied model stores the table once, because the checkpoint
saves the model's state. In exchange, tied and untied checkpoints differ in
structure: the tied one has no head kernel, so loading it into an untied
model is a mismatch that we must resolve explicitly.

What happens to the gradients? During backpropagation each use of the tensor
contributes a gradient, and the contributions add up in the gradient of the
one shared tensor. Let $\mathbf{W}_\textrm{emb}$ and $\mathbf{W}_\textrm{head}$
be the two tables of the untied model. If both hold the values of the tied
table $\mathbf{W}$, the two models compute the same function, and the chain
rule gives

$$\frac{\partial L}{\partial \mathbf{W}} =
\frac{\partial L}{\partial \mathbf{W}_\textrm{emb}} +
\frac{\partial L}{\partial \mathbf{W}_\textrm{head}}.$$

The tied gradient is the sum of the untied gradients. We check this by
loading the tied model's values into the untied one and running the same
backward pass through both:

In [17]:
untied.emb.embedding[...] = lm.emb.embedding
untied.body.kernel[...] = lm.body.kernel
untied.body.bias[...] = lm.body.bias
untied.head.kernel[...] = lm.emb.embedding.T
g_tied = nnx.grad(lambda model: model(tokens).sum())(lm)
g_untied = nnx.grad(lambda model: model(tokens).sum())(untied)
jnp.allclose(g_tied['emb']['embedding'],
             g_untied['emb']['embedding'] + g_untied['head']['kernel'].T)

Array(True, dtype=bool)

### Frozen Parameters

Fine-tuning adapts a model trained on one task to another
(that section). A common recipe keeps the pretrained layers
fixed and trains only a new output layer, which needs little data and little
memory. Keeping a parameter fixed is called *freezing* it.

NNX parameters carry no flag; a parameter is frozen when it is left out of
differentiation and out of the optimizer. A *filter*, a predicate on the
path and the type of each variable, selects the trainable parameters, and
both consumers take it: `nnx.grad` with `argnums=nnx.DiffState(0,
trainable)` differentiates with respect to the selected parameters only,
and `nnx.Optimizer(..., wrt=trainable)` keeps state for them and updates
only them. On a fresh copy of our residual MLP, the filter that selects the
output layer, `layers[3]`, leaves 650 of 18,634 parameters trainable:

In [18]:
finetune = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                          ResidualBlock(64, rngs=rngs),
                          ResidualBlock(64, rngs=rngs),
                          nnx.Linear(64, 10, rngs=rngs))

trainable = nnx.All(nnx.Param, lambda path, _: path[:2] == ('layers', 3))

def count(state):
    return sum(x.size for x in jax.tree.leaves(state))

n_train = count(nnx.state(finetune, trainable))
(n_train, count(nnx.state(finetune, nnx.Param)))

(650, 18634)

The optimizer and `nnx.grad` receive the same filter. One Adam step then
moves the head and nothing else:

In [19]:
head_opt = nnx.Optimizer(finetune, optax.adam(0.1), wrt=trainable)
before = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
grads = nnx.grad(lambda model: model(X).sum(),
                 argnums=nnx.DiffState(0, trainable))(finetune)
head_opt.update(finetune, grads)
stepped = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
jax.tree_util.tree_map(lambda a, b: bool((a != b).any()), before, stepped)

{'layers': {0: {'bias': False, 'kernel': False},
  1: {'body': {'layers': {0: {'bias': False, 'kernel': False},
     2: {'bias': False, 'kernel': False}}}},
  2: {'body': {'layers': {0: {'bias': False, 'kernel': False},
     2: {'bias': False, 'kernel': False}}}},
  3: {'bias': True, 'kernel': True}}}

Only the two leaves under `layers[3]`, the head's kernel and bias, changed.
The frozen layers received no gradient at all: `grads` holds the two leaves
of the head and nothing else. Because `head_opt` was built over the same
filter, it holds Adam's moments for the 650 trainable parameters only.
Freezing has two more effects that raise no error. First, it does not
reclaim optimizer state that already exists. A freeze in NNX is a filter
given to a new optimizer. An optimizer built over all parameters keeps its
moments, 8 bytes per parameter, for as long as it is kept. The optimizer
`optimizer`, which took one step on `net` when we counted its moments above,
still holds all of them:

In [20]:
def moments(opt):
    return sum(x.size for x in jax.tree.leaves(opt.opt_state)
               if hasattr(x, 'ndim') and x.ndim > 0)

moments(head_opt) == 2 * n_train, moments(optimizer) == 2 * n

(True, True)

Second, freezing governs `nnx.Param` state only. Batch normalization keeps
its running statistics as `nnx.BatchStat` variables and updates them in a
training-mode forward pass. No gradient and no optimizer is involved: the
forward pass alone moves the statistics, so leaving `scale` and `bias` out
of the trainable filter cannot stop it:

In [21]:
bn = nnx.BatchNorm(4, use_running_average=False, momentum=0.9,
                   use_fast_variance=False, rngs=rngs)
before = bn.mean[...]
bn(jax.random.normal(rngs(), (8, 4)) + 3)   # training mode
bool(jnp.allclose(bn.mean, before))

False

The trainable filter and `use_running_average` are independent switches:
the first stops parameter updates, the second stops the forward pass from
updating running statistics. To keep a BatchNorm layer fixed during
fine-tuning we need both. `nnx.view(model, use_running_average=True)`
switches the mode on a view of the whole model.

### Weight Averages

A last kind of state is derived from the parameters but not trained: an *exponential moving average* (EMA) of the weights, kept next to
the trained ones and used for evaluation. After every optimizer step, the
average $\bar{\mathbf{w}}$ moves a fraction of the way toward the current
weights $\mathbf{w}$,

$$\bar{\mathbf{w}} \leftarrow 0.9\, \bar{\mathbf{w}} + 0.1\, \mathbf{w},$$

so it lags behind the iterates and smooths out their fluctuations. Whether
it evaluates better than the last iterate depends on the decay and on the
training schedule; that section measures this. Like
BatchNorm statistics, the average is state that changes outside
backpropagation and belongs in the checkpoint. We start the average at the
current weights and continue the fine-tuning above for five steps.

In optax the average is one more pure function of two state trees:
`optax.incremental_update(new, old, step_size)` returns
`step_size * new + (1 - step_size) * old` for every leaf:

In [22]:
avg = stepped   # start from the current weights
for _ in range(5):
    grads = nnx.grad(lambda model: model(X).sum(),
                     argnums=nnx.DiffState(0, trainable))(finetune)
    head_opt.update(finetune, grads)
    current = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
    avg = optax.incremental_update(current, avg, step_size=0.1)
float(jnp.abs(avg['layers'][3]['bias']
              - current['layers'][3]['bias']).max())

0.36855626106262207

The printed number is the gap between the averaged and the current bias of
the head, and it is 0.3686 on every run. We can predict it. The loss is the
sum of the outputs, so the gradient of each bias entry is the batch size,
2, at every step. Adam divides its first moment by the
square root of its second, so a gradient that never changes moves each entry
by the learning rate, 0.1, per step. If the weights move by 0.1 per step, the
gap $e$ between the average and the weights obeys $e \leftarrow 0.9\,(e +
0.1)$, which starts at 0 and reaches $0.3686$ after five steps. The average
carries a memory of earlier weights that the weights themselves have lost,
so a checkpoint that dropped it could not resume the average where it
stopped.

We can now find every tensor of a model, tell parameters from other state,
count what training stores per parameter, share one tensor between layers,
and exclude parts of a model from training. Every one of these tensors
started from a value that some rule chose. The next section asks which rule
to choose, and how to apply it over the tree.

## Summary

A model's state is a tree of named tensors, and registration decides which
consumers see each one: the optimizer sees the parameters, while checkpoints
and device moves also see buffers, persistent tensors such as running
statistics that receive no gradient. Unregistered tensors are left behind.
Training with Adam in fp32 stores 16 bytes per parameter before any
activation: weight, gradient, and two moments. For an 8-billion-parameter
model that is 128 GB. Tying gives two layers one tensor, which is counted
and updated once and receives the sum of the gradients of its uses.
Freezing keeps parameters from changing, but it does not free optimizer
state that already exists, and it does not stop the updates of running
statistics in training mode.

## Exercises

1. [code] **Running statistics.** In training mode a batch normalization
   layer normalizes with the minibatch mean
   $\hat{\boldsymbol{\mu}}_{\mathcal{B}}$ and variance, and each training
   step updates the running mean $\bar{\boldsymbol{\mu}}$ by

   $$\bar{\boldsymbol{\mu}} \leftarrow \rho\, \bar{\boldsymbol{\mu}} + (1 - \rho)\, \hat{\boldsymbol{\mu}}_{\mathcal{B}}.$$

    1. A fresh layer starts from $\bar{\boldsymbol{\mu}} = \mathbf{0}$.
       Predict $\bar{\boldsymbol{\mu}}$ after the layer has processed $k$
       batches in training mode, each with minibatch mean 3 in every
       feature. Check the prediction for $k = 1, 5, 20$ on a fresh layer
       over 4 features, without any optimizer step, and read off the value
       of $\rho$ the layer uses by default.
    1. Suppose you registered $\bar{\boldsymbol{\mu}}$ as a trainable
       parameter and removed the update above. What gradient does the
       training loss send to $\bar{\boldsymbol{\mu}}$? How do the training
       loss and the evaluation accuracy then compare with those of an
       unmodified model?
    1. Find a loss in $\bar{\boldsymbol{\mu}}$ alone whose gradient step
       with learning rate $\eta$ on each minibatch reproduces the removed
       update, and state the $\eta$ that matches $\rho$.

1. **Parameters and buffers.** that section separates
   trained parameters from persistent state that receives no gradient.
   State a one-line rule that classifies every entry of a model's state as
   one or the other, and check it against the state of `whiten` in
   that section and against the batch normalization
   layer `bn` of that section, including any integer counter
   it keeps.

1. [code] **Memory footprint.** Write a helper that reports, for each
   top-level block of `net`, its number of state entries and its bytes
   under fp32 Adam training with every parameter trained.
    1. Which block dominates? At what output width would the output layer
       overtake one residual block?
    1. Insert a batch normalization layer over the 64 hidden features
       between the two residual blocks. Extend the helper so that it costs
       the layer's running statistics correctly, and report each block's
       bytes per entry. For which blocks is the share of the total bytes
       larger than the share of the total entries, and for which is it
       smaller? Explain both directions.

4. [code] **Tied parameters.** Perturb `untied.head.kernel` so that the two
   tables of the untied twin no longer agree. Then build a tied `ToyLM`
   and fill it with values from `untied`. Which values must the conversion
   discard or combine? Compare the logits of the converted model on
   `tokens` with those of `untied`, and state when the converted model
   reproduces `untied` exactly.

5. [code] **Tying and freezing.** Consider the tied `ToyLM` of this
   section.
    1. Leave the embedding table out of the trainable filter, for example
       with `nnx.All(nnx.Param, nnx.Not(nnx.PathContains('emb')))`, and
       pass the filter to `nnx.grad` and `nnx.Optimizer`. How many
       parameters are trained, and can the head still adapt? What does
       this show about the interaction between tying and freezing?
    1. Starting from the values of `lm`, build a model whose embedding stays
       fixed while its head adapts, and verify both properties after one
       update. How many parameters does it store and how many does it
       train, compared with a fully trainable `lm`?

6. **Fine-tuning memory.** A language model has 8 billion parameters. Its
   output layer is a $4096 \times 128{,}256$ matrix, not tied to the
   embedding. All state is fp32, and the optimizer is Adam.
    1. How many bytes do weights, gradients, and Adam's moments take when
       every parameter is trained? How many 80 GB GPUs does that state
       alone fill?
    1. Freeze everything except the output layer, and build the optimizer
       over the output layer only. How many bytes does the persistent state
       take now, and does it fit on one 80 GB GPU?
    1. Suppose instead that an optimizer built over all parameters took
       one step before the freeze and is kept afterwards. How many bytes
       remain allocated, and does the state still fit?

[Discussions](https://d2l.discourse.group/t/36753)